# Approach 2

Approach 2 differs from Approach 1, in the sense that the derived reactions from Rhea are used as the grunnlag for the final dataframe. Reactions will be conected to their relevant proteins in UniProt, before they are mapped to the transporters of TCDB.

Semantically, it will follow something along these lines: R:Data + RID -> UID -> TCID + substrate.

In [2]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd
import numpy as np

 By 24.02.2025, Rhea contained 17422 reactions. These lay the foundation for the data set./
 However, only 1513 of them are transport reactions, according to Rhea. Despite this, all reactions from Rhea are included, in hopes that more transporters will be connected than from transport reactions alone.

In [ ]:
rhea = pd.read_csv("../Rhea/Rhea.tsv", sep="\t")
rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
rhea.rename(columns={"Reaction identifier" : "RID"}, inplace=True)

,RID,Equation,ChEBI name,ChEBI identifier
0,RHEA:23128,4-hydroxybutanoate + ATP + CoA = 4-hydroxybuta...,4-hydroxybutanoate;ATP;CoA;4-hydroxybutanoyl-C...,CHEBI:16724;CHEBI:30616;CHEBI:57287;CHEBI:5857...
1,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...
2,RHEA:23136,a primary amine + S-adenosyl-L-methionine = a ...,a primary amine;S-adenosyl-L-methionine;a meth...,CHEBI:65296;CHEBI:59789;CHEBI:131823;CHEBI:578...
3,RHEA:23140,beta-nicotinamide D-ribonucleotide + H2O = D-r...,beta-nicotinamide D-ribonucleotide;H2O;D-ribos...,CHEBI:14649;CHEBI:15377;CHEBI:78346;CHEBI:1715...
4,RHEA:23144,N-acetyl-alpha-D-glucosamine + NAD(+) + H2O = ...,N-acetyl-alpha-D-glucosamine;NAD(+);H2O;N-acet...,CHEBI:44278;CHEBI:57540;CHEBI:15377;CHEBI:3843...
...,...,...,...,...
17417,RHEA:23108,"D-arabinono-1,4-lactone + H2O = D-arabinonate ...","D-arabinono-1,4-lactone;H2O;D-arabinonate;H(+)",CHEBI:16292;CHEBI:15377;CHEBI:16157;CHEBI:15378
17418,RHEA:23112,6-methylsalicylate + H(+) = 3-methylphenol + CO2,6-methylsalicylate;H(+);3-methylphenol;CO2,CHEBI:36658;CHEBI:15378;CHEBI:17231;CHEBI:16526
17419,RHEA:23116,4-hydroxybenzoate + ATP + CoA = 4-hydroxybenzo...,4-hydroxybenzoate;ATP;CoA;4-hydroxybenzoyl-CoA...,CHEBI:17879;CHEBI:30616;CHEBI:57287;CHEBI:5735...
17420,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...


Now, one can obtain the RID-UID mappings.

In [4]:
uid_rid_map = pd.read_csv("../UniProt/Modified_queries/RID_UID.tsv", sep="\t")
uid_rid_map["RID"] = uid_rid_map["RID"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)
df = rhea.merge(uid_rid_map, on="RID", how="left")
df

,RID,Equation,ChEBI name,ChEBI identifier,UID
0,RHEA:23128,4-hydroxybutanoate + ATP + CoA = 4-hydroxybuta...,4-hydroxybutanoate;ATP;CoA;4-hydroxybutanoyl-C...,CHEBI:16724;CHEBI:30616;CHEBI:57287;CHEBI:5857...,A4YDR9
1,RHEA:23128,4-hydroxybutanoate + ATP + CoA = 4-hydroxybuta...,4-hydroxybutanoate;ATP;CoA;4-hydroxybutanoyl-C...,CHEBI:16724;CHEBI:30616;CHEBI:57287;CHEBI:5857...,A4YDT1
2,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0A2S0WMS1
3,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0AAX2Q740
4,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0A7Z9D521
...,...,...,...,...,...
36721146,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,A0A8S0L9L7
36721147,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,C6KJY2
36721148,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,G8E656
36721149,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,J7KEZ2


Okay, this df is BIG. No need to make it bigger than necessary by attaching RSIDs yet. The first step is to connecting the UIDs possible to TCIDs.\
For this, one needs to obtain the current data from TCDB. And why not just take it all at once, as it is needed later on?

In [5]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def parse_data(uid_txt, substrates_txt, aa_txt, rsid_txt):

    # TCID and Accesion ID (UID/RefSeq)
    uid_data = [line.split("\t") for line in uid_txt.strip().split("\n")]
    df_uid = pd.DataFrame(uid_data, columns=["UID", "TCID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
                        for line in substrates_lines
                        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    # AA sequence (TCID, UID and AA)
    fasta_io = StringIO(aa_txt)
    tc_data = [[record.description.split("|")[3].split()[0],  # TCID
                record.description.split("|")[2],  # UID
                str(record.seq)]  # AA
                for record in SeqIO.parse(fasta_io, "fasta")]
    
    df_aa = pd.DataFrame(tc_data, columns=["TCID", "UID", "AA"])

    # TCID and RSID
    rsid_data = [[line.split("\t")[0], line.split("\t")[1]]
                 for line in rsid_txt.strip().split("\n")]
    df_rsid = pd.DataFrame(rsid_data, columns=["RSID", "TCID"])

    return df_uid, df_substrates, df_aa, df_rsid

In [6]:
tc_uid_url = "https://www.tcdb.org/cgi-bin/projectv/public/acc2tcid.py"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"
tc_aa_url = "https://www.tcdb.org/public/tcdb"
tc_rsid_url = "https://www.tcdb.org/cgi-bin/projectv/public/refseq.py"

tc_uid_txt = fetch_data(tc_uid_url)
tc_substrates_txt = fetch_data(tc_substrates_url)
tc_aa_txt = fetch_data(tc_aa_url)
tc_rsid_txt = fetch_data(tc_rsid_url)

df_uid, df_substrates, df_aa, df_rsid = parse_data(tc_uid_txt, tc_substrates_txt, tc_aa_txt, tc_rsid_txt)

Now merging df with df_uid on the column UID.

In [ ]:
df = df.merge(df_uid, on="UID", how="left")

,RID,Equation,ChEBI name,ChEBI identifier,UID,TCID
0,RHEA:23128,4-hydroxybutanoate + ATP + CoA = 4-hydroxybuta...,4-hydroxybutanoate;ATP;CoA;4-hydroxybutanoyl-C...,CHEBI:16724;CHEBI:30616;CHEBI:57287;CHEBI:5857...,A4YDR9,NaN
1,RHEA:23128,4-hydroxybutanoate + ATP + CoA = 4-hydroxybuta...,4-hydroxybutanoate;ATP;CoA;4-hydroxybutanoyl-C...,CHEBI:16724;CHEBI:30616;CHEBI:57287;CHEBI:5857...,A4YDT1,NaN
2,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0A2S0WMS1,NaN
3,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0AAX2Q740,NaN
4,RHEA:23132,hydrogen sulfide + 6 oxidized [2Fe-2S]-[ferred...,hydrogen sulfide;[2Fe-2S](2+);H2O;sulfite;[2Fe...,CHEBI:29919;CHEBI:33737;CHEBI:15377;CHEBI:1735...,A0A7Z9D521,NaN
...,...,...,...,...,...,...
36721178,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,A0A8S0L9L7,NaN
36721179,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,C6KJY2,NaN
36721180,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,G8E656,NaN
36721181,RHEA:23124,2 (2R)-3-phosphoglycerate + 2 H(+) = D-ribulos...,"(2R)-3-phosphoglycerate;H(+);D-ribulose 1,5-bi...",CHEBI:58272;CHEBI:15378;CHEBI:57870;CHEBI:1652...,J7KEZ2,NaN


Mapping UIDs to their RSIDs from UniProt/Modified_queries/UID_RSID.tsv.

In [8]:
uid_rsid_map = pd.read_csv("../UniProt/Modified_queries/UID_RSID.tsv", sep="\t")
df = df.merge(uid_rsid_map, on="UID", how="left")

Now, one can map the RSIDs to the missing TCIDs through df_rsid.

In [ ]:
df = df.merge(df_rsid, on="RSID", how="left", suffixes=("_x", "_y"))
df["TCID"] = df["TCID_x"].fillna(df["TCID_y"])
df.drop(columns=["TCID_x", "TCID_y"], inplace=True)
df = df.dropna(subset=["TCID"]).reset_index(drop=True)

Now, it is fitting to map the data from TCID to its belonging CHEBI ids, in order to try to spot any differences!

In [ ]:
df = df.merge(df_substrates, on="TCID", how="left")
df = df.merge(df_aa.iloc[:, [0, 2]], on=["TCID"], how="left")
df["UID"] = df["UID"].fillna(df["RSID"])
df.drop(columns=["RSID"], inplace=True)

Well, this is only a preliminary first draft of the dataset for Approach 2. I need som e more thinking and consideration, before I can agree with myself that this is both correct, enough, and of a decent quality. But now it's the end of the work day. I'm off. Bye.

Btw, remember to go through the different queries in UniProt, assess them. Remove the ones deemed redundant.

In [30]:
df

,RID,Equation,ChEBI name,ChEBI identifier,UID,TCID,CHEBI ID,CHEBI Name,AA
0,RHEA:23172,L-seryl-[3-hydroxy-3-methylglutaryl-coenzyme A...,L-serine residue;ATP;O-phospho-L-serine residu...,CHEBI:29999;CHEBI:30616;CHEBI:83421;CHEBI:4562...,P54646,8.A.104.1.1,NaN,NaN,MAEKQKHDGRVKIGHYVLGDTLGVGTFGKVKIGEHQLTGHKVAVKI...
1,RHEA:23532,cholate + ATP + CoA = choloyl-CoA + AMP + diph...,cholate;ATP;CoA;choloyl-CoA;AMP;diphosphate,CHEBI:29747;CHEBI:30616;CHEBI:57287;CHEBI:5737...,Q9Y2P5,4.C.1.1.13,CHEBI:4984,fatty acid,MGVRQQLALLLLLLLLLWGLGQPVWPVAVALTLRWLLGDPTCCVLL...
2,RHEA:23532,cholate + ATP + CoA = choloyl-CoA + AMP + diph...,cholate;ATP;CoA;choloyl-CoA;AMP;diphosphate,CHEBI:29747;CHEBI:30616;CHEBI:57287;CHEBI:5737...,Q9Y2P5,4.C.1.1.13,CHEBI:4984,fatty acid,MGVRQQLALLLLLLLLLWGLGQPVWPVAVALTLRWLLGDPTCCVLL...
3,RHEA:23532,cholate + ATP + CoA = choloyl-CoA + AMP + diph...,cholate;ATP;CoA;choloyl-CoA;AMP;diphosphate,CHEBI:29747;CHEBI:30616;CHEBI:57287;CHEBI:5737...,Q4LDG0,4.C.1.1.8,CHEBI:4984,fatty acid,MGIWKKLTLLLLLLLLVGLGQPPWPAAMALALRWFLGDPTCLVLLG...
4,RHEA:23680,a ribonucleoside 5'-triphosphate + H2O = a rib...,a ribonucleoside 5'-triphosphate;H2O;a ribonuc...,CHEBI:61557;CHEBI:15377;CHEBI:57930;CHEBI:4347...,P03305,1.A.85.1.8,CHEBI:39124,calcium ion,MNTTDCFIALVQAIREIKALFLSRTTGKMELTLYNGEKKTFYSRPN...
...,...,...,...,...,...,...,...,...,...
59188,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...,P45094,3.A.1.5.27,CHEBI:5437,glutathione,MKLKATLTLAAATLVLAACDQSSSANKSTAQTEAKSSSNNTFVYCT...
59189,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...,P45094,3.A.1.5.27,CHEBI:5437,glutathione,MTNEVKENTPLLNAIGLKKYYPVKKGLFAKPQQVKALDGVSFQLER...
59190,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...,P45094,3.A.1.5.27,CHEBI:5437,glutathione,MALLDVKELSVHFGDKKTPFKAVDRISYQVAQGEVLGIVGESGSGK...
59191,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...,P45094,3.A.1.5.27,CHEBI:5437,glutathione,MFKFVFKRILMVIPTFIAITLITFALVHFIPGDPVEIMMGERGLTA...
